# WF4 - Complete workflow: OptimisationSimulation (ACCIM custom model)

WF4 provides the complete custom-model optimization path, from objective setup to Pareto analysis and decision support.

In practical terms, we will move through preparation, configuration, execution, and verification so every transition in the workflow stays explicit.
As the notebook progresses, each code cell is introduced with context first, then the call itself, and finally quick guidance on how to interpret the result.

Workflow roadmap:
- Preparation: imports, path configuration, and baseline model loading.
- Configuration: output selection, parameter definitions, and problem setup.
- Execution and review: simulation or optimization calls followed by targeted checks.

Main classes, methods, and functions used in this notebook:
- `OptimisationSimulation`
- `OptimisationSimulation.run_optimisation`
- `OptimisationSimulation.plot_best_compromise_solutions`
- `OptimisationSimulation.run_clustering`
- `ef.get_building`
- `optim.discover_available_outputs`
- `optim.select_outputs`
- `ValueError`
- `optim.apply_outputs_preflight`
- `optim.set_outputs_for_simulation`

Narrative structure used in this notebook:
- Each code cell is preceded by an explanatory markdown block.
- Section headings are contextual to the current notebook and execution block.
- Key arguments are described before they are used in code.
- Execution-oriented cells may include a follow-up markdown note to help interpret results.

## WF4 custom optimization: import workflow dependencies

We start by loading every dependency required by the full workflow, so the remaining sections can run without hidden imports.

In this complete workflow notebook, imports are intentionally grouped in one place. That makes reruns in a fresh kernel predictable and easier to debug when versions change.

What to verify right after running this cell:
- Review execution output and proceed when values look consistent with the workflow intent.

In [ ]:
from pathlib import Path

import pandas as pd
import accim
from besos import eppy_funcs as ef
from accim.parametric_and_optimisation.main import OptimisationSimulation

pd.set_option("display.max_columns", 160)


## WF4 custom optimization: estimate expected simulation workload

Before launching the run, we estimate workload size from algorithm budget and scenario multiplicators.

This quick estimate helps avoid oversized jobs and is the right place to adjust evaluation budget if needed.

Variables created in this cell:
- `PROJECT_ROOT`: reused in subsequent workflow blocks.
- `idf_list`: reused in subsequent workflow blocks.
- `epw_list`: reused in subsequent workflow blocks.
- `RESULTS_DIRNAME`: reused in subsequent workflow blocks.
- `EVALUATIONS`: reused in subsequent workflow blocks.
- `POPULATION_SIZE`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Compare estimated run count with your available compute/time budget.
- Adjust evaluations, population size, or climate count if the estimate is too high.

In [ ]:
PROJECT_ROOT = Path.cwd()

idf_list = [
    "OSM_TestResidentialUnit_v01_onlygeometry_SchNatVent_v2520_high_performance.idf",
    "OSM_TestResidentialUnit_v01_onlygeometry_SchNatVent_v2520_low_performance.idf",
]
epw_list = ["Seville.epw", "Sydney.epw"]
RESULTS_DIRNAME = "nb14_main_optim_custom_results_without_idf_loop"

EVALUATIONS = 6
POPULATION_SIZE = 3

print("IDF files found:", [(PROJECT_ROOT / idf).exists() for idf in idf_list])
print("EPW files found:", [(PROJECT_ROOT / epw).exists() for epw in epw_list])


## WF4 custom optimization: define objective and auxiliary analysis output groups

We now separate optimization objectives from auxiliary analysis variables.

This separation is critical for interpretation: objective groups drive solver behavior, while auxiliary groups support diagnostics and comfort/energy storytelling after the run.

Variables created in this cell:
- `objective_meter_names`: reused in subsequent workflow blocks.
- `objective_variable_prefix`: reused in subsequent workflow blocks.
- `analysis_meter_names`: reused in subsequent workflow blocks.
- `analysis_variable_patterns`: reused in subsequent workflow blocks.
- `PARETO_SEPARATE_BY_EPW`: reused in subsequent workflow blocks.
- `PARETO_SEPARATE_BY_IDF`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm objective output names are distinct from auxiliary analysis variables.
- Check the declared target groups before output selection and problem setup.

In [ ]:
objective_meter_names = ["Heating:Electricity", "Cooling:Electricity"]
objective_variable_prefix = "Comfortable Hours_No Applicability"
analysis_meter_names = ["Electricity:HVAC"]
analysis_variable_patterns = [
    "Zone Operative Temperature",
    "Zone Thermal Comfort Fanger Model PMV",
    "Running Average Outdoor Air Temperature",
    "Setpoint Temperature_No Tolerance",
]
PARETO_SEPARATE_BY_EPW = True
PARETO_SEPARATE_BY_IDF = False


## WF4 custom optimization: load the baseline IDF model

Load the base building model used by subsequent runs.

This is where the baseline IDF is loaded for all downstream operations. Later cells reuse this object for output setup, parameterization, and execution.

Key arguments used in this cell and why they matter:
- `idf_object`: defines this setting for the current call.
- `timesteps`: defines this setting for the current call.

Variables created in this cell:
- `building`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm these objects are available for the next block: `building`.

In [ ]:
def _load_and_prepare_building(idf_name: str):
    print("\nPreparing IDF:", idf_name)
    building = ef.get_building(idf_name)
    accim.utils.reduce_runtime(idf_object=building, timesteps=2)
    return building


## WF4 custom optimization: build the model collection for multi-building runs

Assemble the list of building models used in multi-building execution.

This block assembles one or more building objects for multi-building execution. The resulting collection is reused by the simulation or optimization object.

Variables created in this cell:
- `buildings`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Verify the number of building objects matches the intended multi-IDF scenario.
- Check each building entry before creating the simulation/optimization object.

In [ ]:
buildings = [_load_and_prepare_building(idf_name) for idf_name in idf_list]


## WF4 custom optimization: initialize optimization simulation object

Here we instantiate the optimization engine that will evaluate candidates across the configured climates and models.

Once initialized, the object becomes the central container for outputs, objectives, and algorithm execution settings.

Key arguments used in this cell and why they matter:
- `buildings`: defines this setting for the current call.
- `parameters_type`: parameter family used in the simulation setup.
- `epws`: list of EPW weather files used to run scenarios.
- `output_freqs`: time frequency requested for outputs.

Variables created in this cell:
- `optim`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm these objects are available for the next block: `optim`.

In [ ]:
optim = OptimisationSimulation(
    buildings=buildings,
    parameters_type="accim custom model",
    epws=epw_list,
    output_freqs=["Hourly"],
)


## WF4 custom optimization: discover available meter and variable outputs

Before locking outputs, we scan what this model setup can actually report.

Doing this now prevents downstream failures caused by requesting unavailable variables or meter names.

Key arguments used in this cell and why they matter:
- `reduce_sim_time`: enables a fast scan mode when discovering outputs.
- `prefer`: preferred backend for output discovery.
- `idf_scope`: defines this setting for the current call.

Variables created in this cell:
- `available_outputs`: reused in subsequent workflow blocks.
- `outputs_meta`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Check that the discovered meter/variable tables are not unexpectedly empty.
- Confirm the reported discovery source matches your preferred backend.

In [ ]:
available_outputs = optim.discover_available_outputs(
    reduce_sim_time=True,
    prefer="testsimeplus",
    idf_scope="first",
)
outputs_meta = available_outputs["meta"]
print("Discovery source:", outputs_meta.get("source"))


## WF4 custom optimization: select objective and analysis outputs

We now select the objective and auxiliary outputs that will feed execution and analysis.

Think of this as output curation: this cell narrows the available catalog to the exact metrics needed downstream.

Key arguments used in this cell and why they matter:
- `meters`: energy meters to include as outputs.
- `match`: matching rule used to select outputs.
- `on_missing`: behavior when a requested output is missing.
- `suggest`: shows suggestions when no exact match is found.
- `idf_scope`: defines this setting for the current call.
- `variables`: output variables to include by name or pattern.
- `subset`: defines this setting for the current call.
- `ignore_index`: defines this setting for the current call.
- `drop`: defines this setting for the current call.

Variables created in this cell:
- `selection_meters_obj`: reused in subsequent workflow blocks.
- `df_meters_obj`: reused in subsequent workflow blocks.
- `selection_vars_obj`: reused in subsequent workflow blocks.
- `df_vars_obj_candidates`: reused in subsequent workflow blocks.
- `df_vars_obj`: reused in subsequent workflow blocks.
- `selection_meters_analysis`: reused in subsequent workflow blocks.
- `df_meters_analysis`: reused in subsequent workflow blocks.
- `selection_vars_analysis`: reused in subsequent workflow blocks.
- `df_vars_analysis`: reused in subsequent workflow blocks.
- `df_meters_sel`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm the selected tables include all objective outputs you plan to track.
- Verify preflight did not report blocking missing outputs.

In [ ]:
selection_meters_obj = optim.select_outputs(
    meters=objective_meter_names,
    match="case_insensitive",
    on_missing="raise",
    suggest=True,
    idf_scope="first",
)
df_meters_obj = selection_meters_obj["meters"]

selection_vars_obj = optim.select_outputs(
    variables=[objective_variable_prefix],
    match="contains",
    on_missing="raise",
    suggest=True,
    idf_scope="first",
)
df_vars_obj_candidates = selection_vars_obj["variables"]
if df_vars_obj_candidates.empty:
    raise ValueError("No objective Output:Variable was found for maximization.")
if "variable_name" not in df_vars_obj_candidates.columns:
    raise ValueError("The objective-variable selection does not contain the variable_name column.")
df_vars_obj = df_vars_obj_candidates[
    df_vars_obj_candidates["variable_name"].astype(str).str.startswith(objective_variable_prefix)
].copy()
if df_vars_obj.empty:
    raise ValueError(
        f"No objective variables were found starting with '{objective_variable_prefix}'."
    )
print("Objective variables selected for maximization:")
for name in df_vars_obj["variable_name"].tolist():
    print(" -", name)

selection_meters_analysis = optim.select_outputs(
    meters=analysis_meter_names,
    match="case_insensitive",
    on_missing="warn",
    suggest=True,
    idf_scope="first",
)
df_meters_analysis = selection_meters_analysis["meters"]

selection_vars_analysis = optim.select_outputs(
    variables=analysis_variable_patterns,
    match="contains",
    on_missing="warn",
    suggest=True,
    idf_scope="first",
)
df_vars_analysis = selection_vars_analysis["variables"]

def _concat_unique(dfs, subset):
    non_empty = [d for d in dfs if d is not None and not d.empty]
    if not non_empty:
        return pd.DataFrame()
    merged = pd.concat(non_empty, ignore_index=True)
    use_subset = [c for c in subset if c in merged.columns]
    if use_subset:
        merged = merged.drop_duplicates(subset=use_subset)
    else:
        merged = merged.drop_duplicates()
    return merged.reset_index(drop=True)

df_meters_sel = _concat_unique(
    [df_meters_obj, df_meters_analysis],
    subset=["key_name", "frequency"],
)
df_vars_sel = _concat_unique(
    [df_vars_obj, df_vars_analysis],
    subset=["key_value", "variable_name", "frequency"],
)

print("Objective meters:", len(df_meters_obj))
print("Objective variables:", len(df_vars_obj))
print("Analysis-only meters:", len(df_meters_analysis))
print("Analysis-only variables:", len(df_vars_analysis))
print("Total meters requested in files:", len(df_meters_sel))
print("Total variables requested in files:", len(df_vars_sel))


## WF4 custom optimization: run output preflight checks before execution

This block applies preflight checks to the selected outputs before execution.

Preflight is the final schema gate; if this step is clean, output-related runtime errors are much less likely.

Key arguments used in this cell and why they matter:
- `df_meters_sel`: table of selected meters to apply to the IDF.
- `df_vars_sel`: table of selected variables to apply to the IDF.
- `clean_mode`: cleanup mode applied before adding outputs.
- `validate_before_apply`: whether to validate before editing the IDF.
- `validate_after_apply`: whether to validate after editing the IDF.
- `on_missing`: behavior when a requested output is missing.
- `validation_idf_scope`: defines this setting for the current call.

What to verify right after running this cell:
- Confirm the selected tables include all objective outputs you plan to track.
- Verify preflight did not report blocking missing outputs.

In [ ]:
optim.apply_outputs_preflight(
    df_meters_sel=df_meters_sel,
    df_vars_sel=df_vars_sel,
    clean_mode="all",
    validate_before_apply=False,
    validate_after_apply=True,
    on_missing="raise",
    validation_idf_scope="first",
)


## WF4 custom optimization: apply selected output schema to simulation settings

Now we persist the validated output schema in the active workflow object.

After this point, runs use these exact output definitions, so this step should be revisited whenever target metrics change.

Key arguments used in this cell and why they matter:
- `df_output_meter`: final meter table used by the simulation.
- `df_output_variable`: final variable table used by the simulation.

What to verify right after running this cell:
- Confirm output schema assignment completed without warnings.
- Verify target meter/variable names match the intended analysis outputs.

In [ ]:
optim.set_outputs_for_simulation(
    df_output_meter=df_meters_obj,
    df_output_variable=df_vars_obj,
)


## WF4 custom optimization: define parameter ranges

This section defines the decision space and objective direction used in the run.

Treat this as the modeling core of the workflow: parameter ranges and minimization/maximization choices control result meaning.

Key arguments used in this cell and why they matter:
- `accis_params_dict`: ACCIS/aPMV parameter definition used in the study.

Variables created in this cell:
- `accis_parameters`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm these objects are available for the next block: `accis_parameters`.

In [ ]:
accis_parameters = {
    "CustAST_m": (0.20, 0.85),
    "CustAST_n": (9, 21),
    "CustAST_ASToffset": (1.0, 5.0),
    "CustAST_ASTall": (8, 14),
    "CustAST_ASTaul": (30, 37),
}
optim.set_parameters(accis_params_dict=accis_parameters)


This section defines the decision space and objective direction used in the run.

Treat this as the modeling core of the workflow: parameter ranges and minimization/maximization choices control result meaning.

Key arguments used in this cell and why they matter:
- `minimize_outputs`: boolean vector indicating which objectives are minimized.

Variables created in this cell:
- `minimize_outputs`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm these objects are available for the next block: `minimize_outputs`.

In [ ]:
minimize_outputs = [True] * len(df_meters_obj) + [False] * len(df_vars_obj)
print("minimize_outputs vector:", minimize_outputs)
optim.set_problem(minimize_outputs=minimize_outputs)


## WF4 custom optimization: estimate expected simulation workload

Before launching the run, we estimate workload size from algorithm budget and scenario multiplicators.

This quick estimate helps avoid oversized jobs and is the right place to adjust evaluation budget if needed.

Key arguments used in this cell and why they matter:
- `evaluations`: total number of optimization evaluations.
- `population_size`: population size for the evolutionary algorithm.
- `epws`: list of EPW weather files used to run scenarios.

Variables created in this cell:
- `estimated`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Compare estimated run count with your available compute/time budget.
- Adjust evaluations, population size, or climate count if the estimate is too high.

In [ ]:
estimated = optim.estimate_optimisation_sims(
    evaluations=EVALUATIONS,
    population_size=POPULATION_SIZE,
    epws=epw_list,
)
print("Estimated simulations:", estimated)


## WF4 custom optimization: run NSGA-II optimization and inspect Pareto flags

This cell launches the optimizer with the configured budget and returns the evaluated solution set.

Immediately after the run, review objective columns and Pareto flags to confirm solver behavior.

Key arguments used in this cell and why they matter:
- `epws`: list of EPW weather files used to run scenarios.
- `out_dir`: output folder for generated artifacts and results.
- `evaluations`: total number of optimization evaluations.
- `population_size`: population size for the evolutionary algorithm.
- `algorithm`: optimization algorithm to execute.
- `processes`: number of parallel processes for simulation runs.
- `keep_df`: policy for storing output dataframes.
- `keep_sim_files`: policy for preserving simulation files.
- `keep_sim_files_batch_size`: batch size used when preserving simulation files.
- `pareto_separate_by_epw`: defines this setting for the current call.
- `pareto_separate_by_idf`: defines this setting for the current call.

Variables created in this cell:
- `outputs_optim`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Check objective columns and the `pareto-optimal` indicator before post-analysis.
- Confirm the selected algorithm and evaluation budget were applied as intended.

In [ ]:
outputs_optim = optim.run_optimisation(
    epws=epw_list,
    out_dir=RESULTS_DIRNAME,
    evaluations=EVALUATIONS,
    population_size=POPULATION_SIZE,
    algorithm="NSGAII",
    processes=1,
    keep_df="all",
    keep_sim_files="non-dominated",
    keep_sim_files_batch_size=POPULATION_SIZE,
    pareto_separate_by_epw=PARETO_SEPARATE_BY_EPW,
    pareto_separate_by_idf=PARETO_SEPARATE_BY_IDF,
)


### Reading the result

Inspect objective columns and `pareto-optimal` status before moving to compromise or robustness analysis.

## WF4 custom optimization: preview execution outputs and row counts

We perform a quick sanity check on row counts and first records before post-processing.

This lightweight preview catches missing columns or unexpected schema changes early, while context is still fresh.

What to verify right after running this cell:
- Confirm row counts and first records look consistent with the execution settings.
- Check that expected objective/output columns are present before plotting or MCDM steps.

In [ ]:
print("Combined rows:", len(outputs_optim))
print(outputs_optim.head())


## WF4 custom optimization: normalize datasets for cross-case comparison

In this stage we normalize outputs to make cross-case and cross-climate comparisons fairer.

Keep normalization choices consistent across workflows when preparing comparative figures or decision-support summaries.

Key arguments used in this cell and why they matter:
- `mode`: operation mode for the current method.
- `df_types`: dataset groups that should be normalized.

Variables created in this cell:
- `optim_case`: reused in subsequent workflow blocks.
- `plot_out_dir`: reused in subsequent workflow blocks.
- `optim_df`: reused in subsequent workflow blocks.
- `main_energy_col`: reused in subsequent workflow blocks.
- `secondary_energy_col`: reused in subsequent workflow blocks.
- `plot_energy_vars`: reused in subsequent workflow blocks.
- `x_param`: reused in subsequent workflow blocks.
- `y_param`: reused in subsequent workflow blocks.
- `aux_param`: reused in subsequent workflow blocks.
- `epw_col`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm normalization scope (`df_types`) matches the intended datasets.
- Check one or two normalized columns to verify value scaling behaves as expected.

In [ ]:
optim_case = optim
plot_out_dir = f"{RESULTS_DIRNAME}/plots"

from IPython.display import Image, display
import matplotlib.pyplot as plt

optim_case.set_building_floor_area(mode="all")
optim_case.normalize_outputs(df_types=["optimisation"])
optim_df = optim_case.outputs_optimisation.copy()

main_energy_col = next(c for c in optim_df.columns if "Heating:Electricity" in c)
secondary_energy_col = next(c for c in optim_df.columns if "Cooling:Electricity" in c)
plot_energy_vars = [main_energy_col, secondary_energy_col]

x_param = "CustAST_m"
y_param = "CustAST_n"
aux_param = "CustAST_ASToffset"

epw_col = "epw"
idf_col = "idf"
pareto_col = "pareto-optimal"

box_hue = x_param
scatter_style = pareto_col
lines_hue = epw_col
lines_col = idf_col
heatmap_col = epw_col
distribution_x = epw_col
distribution_hue = pareto_col
radar_metrics = [main_energy_col, secondary_energy_col, x_param, y_param]
radar_group_by = epw_col
pareto_color_by = aux_param
pareto_size_by = x_param


## WF4 custom optimization: generate categorical boxplots for key metrics

We start the visualization stage with categorical boxplots to compare distribution spread across key parameter groupings.

This view is useful for quickly spotting median shifts and variability differences between climates or control settings.

Key arguments used in this cell and why they matter:
- `df_source`: defines this setting for the current call.
- `y_vars`: list of output variables to plot.
- `col`: facet column used in plotting.
- `row`: defines this setting for the current call.
- `hue`: field used to color plot traces.
- `out_dir`: output folder for generated artifacts and results.
- `normalize_per_m2`: whether results are normalized by floor area.
- `show_points`: whether individual points are overlaid on summary plots.
- `sharey`: whether facets share the same y-axis scale.
- `height`: height of each plotting panel.
- `aspect`: aspect ratio of each plotting panel.

Variables created in this cell:
- `fig_boxplots`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Open exported figures and verify labels, grouping, and normalization settings.
- Confirm output files were saved in the expected folder.

In [ ]:
fig_boxplots = optim_case.plot_categorical_boxplots(
    df_source="optimisation",
    y_vars=plot_energy_vars,
    col=epw_col,
    row=None,
    hue=box_hue,
    out_dir=plot_out_dir,
    normalize_per_m2=False,
    show_points=True,
    sharey=True,
    height=3.6,
    aspect=1.1,
)
display(fig_boxplots.fig)
plt.show()


### Reading the result

Open saved figures and verify that filters, grouping, and labels match your intended interpretation.

This cell builds scatter relationships between parameters and key outputs.

Scatter views help detect monotonic trends, nonlinear regions, and potential interaction patterns worth deeper analysis.

Key arguments used in this cell and why they matter:
- `x`: x-axis variable used in plotting.
- `y`: y-axis variable used in plotting.
- `df_source`: defines this setting for the current call.
- `hue`: field used to color plot traces.
- `style`: field used to style plot traces.
- `out_dir`: output folder for generated artifacts and results.
- `normalize_per_m2`: whether results are normalized by floor area.

Variables created in this cell:
- `fig_scatter`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Open exported figures and verify labels, grouping, and normalization settings.
- Confirm output files were saved in the expected folder.

In [ ]:
fig_scatter = optim_case.plot_parametric_scatter(
    x=x_param,
    y=main_energy_col,
    df_source="optimisation",
    hue=epw_col,
    style=scatter_style,
    out_dir=plot_out_dir,
    normalize_per_m2=False,
)
display(fig_scatter.fig)
plt.show()


### Reading the result

Open saved figures and verify that filters, grouping, and labels match your intended interpretation.

## WF4 custom optimization: generate parametric line trends

We now create line plots to inspect how outputs evolve along ordered parameter values.

Line trends make cross-climate and cross-model contrasts easier to read when the x-axis has clear ordering.

Key arguments used in this cell and why they matter:
- `x`: x-axis variable used in plotting.
- `y_vars`: list of output variables to plot.
- `df_source`: defines this setting for the current call.
- `hue`: field used to color plot traces.
- `col`: facet column used in plotting.
- `out_dir`: output folder for generated artifacts and results.
- `normalize_per_m2`: whether results are normalized by floor area.
- `filename`: filename used when saving a figure.

Variables created in this cell:
- `fig_lines`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Open exported figures and verify labels, grouping, and normalization settings.
- Confirm output files were saved in the expected folder.

In [ ]:
fig_lines = optim_case.plot_parametric_lines(
    x=y_param,
    y_vars=plot_energy_vars,
    df_source="optimisation",
    hue=lines_hue,
    col=lines_col,
    out_dir=plot_out_dir,
    normalize_per_m2=False,
)
for plot_path in fig_lines.values():
    display(Image(filename=str(plot_path)))


### Reading the result

Open saved figures and verify that filters, grouping, and labels match your intended interpretation.

Here we generate a heatmap representation of output response across a two-parameter grid.

Heatmaps are practical for identifying favorable parameter regions before narrowing the decision space.

Key arguments used in this cell and why they matter:
- `x`: x-axis variable used in plotting.
- `y`: y-axis variable used in plotting.
- `z`: z-axis or color-value variable used in plotting.
- `df_source`: defines this setting for the current call.
- `col`: facet column used in plotting.
- `out_dir`: output folder for generated artifacts and results.
- `normalize_per_m2`: whether results are normalized by floor area.
- `filename`: filename used when saving a figure.

Variables created in this cell:
- `fig_heatmap`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Open exported figures and verify labels, grouping, and normalization settings.
- Confirm output files were saved in the expected folder.

In [ ]:
fig_heatmap = optim_case.plot_parametric_heatmap(
    x=x_param,
    y=y_param,
    z=main_energy_col,
    df_source="optimisation",
    col=heatmap_col,
    out_dir=plot_out_dir,
    normalize_per_m2=False,
)
display(Image(filename=str(fig_heatmap)))


### Reading the result

Open saved figures and verify that filters, grouping, and labels match your intended interpretation.

Here we build 2D density/hexbin plots to visualize concentration patterns in parameter-output space.

This view is useful for identifying dense performance regions and sparse outliers that may deserve targeted checks.

Key arguments used in this cell and why they matter:
- `x`: x-axis variable used in plotting.
- `y`: y-axis variable used in plotting.
- `kind`: plot kind (for example violin or hexbin).
- `df_source`: defines this setting for the current call.
- `col`: facet column used in plotting.
- `out_dir`: output folder for generated artifacts and results.
- `normalize_per_m2`: whether results are normalized by floor area.
- `filename`: filename used when saving a figure.

Variables created in this cell:
- `fig_density_2d`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Open exported figures and verify labels, grouping, and normalization settings.
- Confirm output files were saved in the expected folder.

In [ ]:
fig_density_2d = optim_case.plot_parametric_density_2d(
    x=aux_param,
    y=main_energy_col,
    kind="hexbin",
    df_source="optimisation",
    col=heatmap_col,
    out_dir=plot_out_dir,
    normalize_per_m2=False,
)
display(Image(filename=str(fig_density_2d)))


### Reading the result

Open saved figures and verify that filters, grouping, and labels match your intended interpretation.

In this section we generate the figures that support interpretation and reporting.

Each plot should be checked for labeling, grouping, and normalization assumptions before sharing results.

Key arguments used in this cell and why they matter:
- `x`: x-axis variable used in plotting.
- `y`: y-axis variable used in plotting.
- `z`: z-axis or color-value variable used in plotting.
- `df_source`: defines this setting for the current call.
- `col`: facet column used in plotting.
- `out_dir`: output folder for generated artifacts and results.
- `normalize_per_m2`: whether results are normalized by floor area.
- `filename`: filename used when saving a figure.

Variables created in this cell:
- `fig_contour`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Open exported figures and verify labels, grouping, and normalization settings.
- Confirm output files were saved in the expected folder.

In [ ]:
fig_contour = optim_case.plot_parametric_contour(
    x=x_param,
    y=y_param,
    z=main_energy_col,
    df_source="optimisation",
    col=heatmap_col,
    out_dir=plot_out_dir,
    normalize_per_m2=False,
)
display(Image(filename=str(fig_contour)))


### Reading the result

Open saved figures and verify that filters, grouping, and labels match your intended interpretation.

## WF4 custom optimization: generate distribution plots for objective variables

This cell generates distribution plots for objective variables to compare spread and shape.

Distribution views complement Pareto plots by showing variability and overlap between parameter groups.

Key arguments used in this cell and why they matter:
- `x`: x-axis variable used in plotting.
- `y_vars`: list of output variables to plot.
- `kind`: plot kind (for example violin or hexbin).
- `df_source`: defines this setting for the current call.
- `hue`: field used to color plot traces.
- `out_dir`: output folder for generated artifacts and results.
- `normalize_per_m2`: whether results are normalized by floor area.
- `show_points`: whether individual points are overlaid on summary plots.
- `filename`: filename used when saving a figure.

Variables created in this cell:
- `fig_distributions`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Open exported figures and verify labels, grouping, and normalization settings.
- Confirm output files were saved in the expected folder.

In [ ]:
fig_distributions = optim_case.plot_parametric_distributions(
    x=distribution_x,
    y_vars=plot_energy_vars,
    kind="violin",
    df_source="optimisation",
    hue=distribution_hue,
    out_dir=plot_out_dir,
    normalize_per_m2=False,
    show_points=True,
)
for plot_path in fig_distributions.values():
    display(Image(filename=str(plot_path)))


### Reading the result

Open saved figures and verify that filters, grouping, and labels match your intended interpretation.

This block adds ECDF plots to compare output distributions across scenarios.

ECDF curves are especially useful when comparing the full distribution rather than only mean or median behavior.

Key arguments used in this cell and why they matter:
- `x`: x-axis variable used in plotting.
- `df_source`: defines this setting for the current call.
- `hue`: field used to color plot traces.
- `out_dir`: output folder for generated artifacts and results.
- `normalize_per_m2`: whether results are normalized by floor area.
- `filename`: filename used when saving a figure.

Variables created in this cell:
- `fig_ecdf`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Open exported figures and verify labels, grouping, and normalization settings.
- Confirm output files were saved in the expected folder.

In [ ]:
fig_ecdf = optim_case.plot_parametric_ecdf(
    x=main_energy_col,
    df_source="optimisation",
    hue="epw",
    out_dir=plot_out_dir,
    normalize_per_m2=False,
)
display(Image(filename=str(fig_ecdf)))


### Reading the result

Open saved figures and verify that filters, grouping, and labels match your intended interpretation.

## WF4 custom optimization: generate radar comparison across grouped solutions

Here we build radar charts to compare grouped solutions across multiple metrics.

Radar views are helpful for communication when multiple performance dimensions must be interpreted simultaneously.

Key arguments used in this cell and why they matter:
- `metrics`: metrics included in the radar plot.
- `group_by`: field used to group traces in the radar plot.
- `df_source`: defines this setting for the current call.
- `out_dir`: output folder for generated artifacts and results.
- `normalize_per_m2`: whether results are normalized by floor area.
- `filename`: filename used when saving a figure.

Variables created in this cell:
- `fig_radar`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Open exported figures and verify labels, grouping, and normalization settings.
- Confirm output files were saved in the expected folder.

In [ ]:
fig_radar = optim_case.plot_parametric_radar(
    metrics=radar_metrics,
    group_by=radar_group_by,
    df_source="optimisation",
    out_dir=plot_out_dir,
    normalize_per_m2=False,
)
for plot_path in sorted(Path(plot_out_dir).glob("plot_parametric_radar_optimisation_group_*.png")):
    display(Image(filename=str(plot_path)))
display(fig_radar)


### Reading the result

Open saved figures and verify that filters, grouping, and labels match your intended interpretation.

Here we visualize the Pareto front to inspect objective trade-offs and parameter effects.

This figure is the main optimization diagnostic; confirm non-dominated structure before running decision-support methods.

Key arguments used in this cell and why they matter:
- `color_by`: field used to color Pareto points.
- `size_by`: field used to scale Pareto markers.
- `out_dir`: output folder for generated artifacts and results.
- `normalize_per_m2`: whether results are normalized by floor area.
- `filename`: filename used when saving a figure.

What to verify right after running this cell:
- Open exported figures and verify labels, grouping, and normalization settings.
- Confirm output files were saved in the expected folder.

In [ ]:
optim_case.plot_pareto_front(
    color_by=pareto_color_by,
    size_by=pareto_size_by,
    out_dir=plot_out_dir,
    normalize_per_m2=False,
)
for plot_path in sorted(Path(plot_out_dir).glob("plot_pareto_front_*.png")):
    display(Image(filename=str(plot_path)))


### Reading the result

Open saved figures and verify that filters, grouping, and labels match your intended interpretation.

## WF4 custom optimization: generate parallel-coordinates overview

This block generates parallel-coordinates plots to inspect trade-offs across many dimensions at once.

Use this as a compact overview before selecting candidate solutions for decision-support analysis.

Key arguments used in this cell and why they matter:
- `out_dir`: output folder for generated artifacts and results.
- `filename`: filename used when saving a figure.

What to verify right after running this cell:
- Open exported figures and verify labels, grouping, and normalization settings.
- Confirm output files were saved in the expected folder.

In [ ]:
optim_case.plot_parallel_coordinates(out_dir=plot_out_dir)
for plot_path in sorted(Path(plot_out_dir).glob("plot_parallel_coordinates_*.png")):
    display(Image(filename=str(plot_path)))


### Reading the result

Open saved figures and verify that filters, grouping, and labels match your intended interpretation.

We generate a pairwise scatter matrix to explore bivariate relations between key outputs and parameters.

This diagnostic often reveals collinearity or clusters that are less visible in single-plot views.

Key arguments used in this cell and why they matter:
- `out_dir`: output folder for generated artifacts and results.
- `normalize_per_m2`: whether results are normalized by floor area.
- `filename`: filename used when saving a figure.

What to verify right after running this cell:
- Open exported figures and verify labels, grouping, and normalization settings.
- Confirm output files were saved in the expected folder.

In [ ]:
optim_case.plot_pairwise_scatter_matrix(
    out_dir=plot_out_dir,
    normalize_per_m2=False,
)
for plot_path in sorted(Path(plot_out_dir).glob("plot_pairwise_scatter_matrix_*.png")):
    display(Image(filename=str(plot_path)))


### Reading the result

Open saved figures and verify that filters, grouping, and labels match your intended interpretation.

## WF4 custom optimization: generate compromise-solution comparison plots

This section compares compromise-solution methods in a unified visualization output.

Comparing methods side by side helps you judge stability of recommendations before final selection.

Key arguments used in this cell and why they matter:
- `out_dir`: output folder for generated artifacts and results.
- `normalize_per_m2`: whether results are normalized by floor area.
- `separate_by_epw`: defines this setting for the current call.
- `separate_by_idf`: defines this setting for the current call.
- `filename`: filename used when saving a figure.

Variables created in this cell:
- `fig_mcdm`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Open exported figures and verify labels, grouping, and normalization settings.
- Confirm output files were saved in the expected folder.

In [ ]:
fig_mcdm = optim_case.plot_best_compromise_solutions(
    out_dir=plot_out_dir,
    normalize_per_m2=False,
    separate_by_epw=PARETO_SEPARATE_BY_EPW,
    separate_by_idf=PARETO_SEPARATE_BY_IDF,
)
for plot_path in sorted(Path(plot_out_dir).glob("plot_mcdm_best_solutions*.png")):
    display(Image(filename=str(plot_path)))
display(fig_mcdm)


### Reading the result

Open saved figures and verify that filters, grouping, and labels match your intended interpretation.

This block prepares hourly datasets and shared plotting settings used by the next hourly visualizations.

By centralizing hourly tokens, filters, and categorical ordering here, the following hourly plots remain concise and consistent.

Key arguments used in this cell and why they matter:
- `parents`: defines this setting for the current call.
- `exist_ok`: defines this setting for the current call.
- `output_columns`: output columns requested for hourly extraction.
- `file_source`: data source to read from (csv, parquet, etc.).
- `skip_confirmation`: skips interactive confirmation prompts.
- `start_date`: start timestamp used for time indexing.
- `normalize_per_m2`: whether results are normalized by floor area.
- `df_source`: defines this setting for the current call.
- `value_tokens`: defines this setting for the current call.
- `row`: defines this setting for the current call.
- `col`: facet column used in plotting.
- `out_dir`: output folder for generated artifacts and results.

Variables created in this cell:
- `hourly_plot_dir`: reused in subsequent workflow blocks.
- `hourly_meter_tokens`: reused in subsequent workflow blocks.
- `hourly_variable_tokens`: reused in subsequent workflow blocks.
- `hourly_output_columns`: reused in subsequent workflow blocks.
- `hourly_value_tokens`: reused in subsequent workflow blocks.
- `hourly_common`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Open exported figures and verify labels, grouping, and normalization settings.
- Confirm output files were saved in the expected folder.

In [ ]:
hourly_plot_dir = Path(plot_out_dir)
hourly_plot_dir.mkdir(parents=True, exist_ok=True)

hourly_meter_tokens = df_meters_sel["key_name"].dropna().astype(str).tolist()
hourly_variable_tokens = df_vars_sel["variable_name"].dropna().astype(str).tolist()
hourly_output_columns = list(dict.fromkeys(hourly_meter_tokens + hourly_variable_tokens))
hourly_value_tokens = hourly_meter_tokens

optim_case.get_hourly_df_optimisation(
    output_columns=hourly_output_columns,
    file_source="csv",
    skip_confirmation=True,
    start_date="2024-01-01 01",
    normalize_per_m2=False,
)

hourly_common = dict(
    df_source="optimisation_hourly",
    value_tokens=hourly_value_tokens,
    row=None,
    col="idf",
    out_dir=str(hourly_plot_dir),
)

print("Hourly output columns requested:", hourly_output_columns)
print("Hourly value tokens:", hourly_value_tokens)
print("Facet row:", None, "| Facet col:", "idf")
print("Hourly shape:", optim_case.outputs_optimisation_hourly.shape)


### Reading the result

Open saved figures and verify that filters, grouping, and labels match your intended interpretation.

## WF4 custom optimization: prepare hourly dataframe and plotting configuration

This block prepares hourly datasets and shared plotting settings used by the next hourly visualizations.

By centralizing hourly tokens, filters, and categorical ordering here, the following hourly plots remain concise and consistent.

Key arguments used in this cell and why they matter:
- `epw_filter`: EPW filter applied in hourly plots.
- `x`: x-axis variable used in plotting.
- `x_label`: x-axis label for exported figures.
- `y_label`: y-axis label for exported figures.
- `filename`: filename used when saving a figure.
- `height`: height of each plotting panel.
- `aspect`: aspect ratio of each plotting panel.
- `marker_size`: defines this setting for the current call.
- `marker_alpha`: defines this setting for the current call.

Variables created in this cell:
- `scatter_grid`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Open exported figures and verify labels, grouping, and normalization settings.
- Confirm output files were saved in the expected folder.

In [ ]:
scatter_grid = optim_case.plot_hourly_scatter(
    epw_filter="Seville",
    x="datetime",
    x_label="Time",
    y_label="Hourly Electricity [J]",
    filename="plot_hourly_scatter_time_seville.png",
    height=2.6,
    aspect=1.05,
    marker_size=0.7,
    marker_alpha=0.55,
    **hourly_common,
)
display(Image(filename=str(scatter_grid.saved_path)))


### Reading the result

Open saved figures and verify that filters, grouping, and labels match your intended interpretation.

This block prepares hourly datasets and shared plotting settings used by the next hourly visualizations.

By centralizing hourly tokens, filters, and categorical ordering here, the following hourly plots remain concise and consistent.

Key arguments used in this cell and why they matter:
- `epw_filter`: EPW filter applied in hourly plots.
- `y_label`: y-axis label for exported figures.
- `filename`: filename used when saving a figure.
- `height`: height of each plotting panel.
- `aspect`: aspect ratio of each plotting panel.

Variables created in this cell:
- `line_grid`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Open exported figures and verify labels, grouping, and normalization settings.
- Confirm output files were saved in the expected folder.

In [ ]:
for city in ["Seville", "Sydney"]:
    line_grid = optim_case.plot_hourly_lines(
        epw_filter=city,
        y_label="Hourly Electricity [J]",
        filename=f"plot_hourly_line_{city.lower()}.png",
        height=2.6,
        aspect=1.05,
        **hourly_common,
    )
    display(Image(filename=str(line_grid.saved_path)))


### Reading the result

Open saved figures and verify that filters, grouping, and labels match your intended interpretation.